# Model Selection Lab
## Grid Search for *k*-NN

To get us started we have an example that fits a *k*-NN model for the `HotelRevHelpfulness` dataset. It assesses three options:
- whether to use a StandardScaler, MinMaxScaler or no scaler. 
- what <em>k</em> to use for <em>k</em>-NN
- what weighting policy

In [1]:
from sklearn.model_selection import GridSearchCV # grisearch
from sklearn.neighbors import KNeighborsClassifier # k-NN classifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler # for normalisation
from sklearn.datasets import load_digits # pi
from sklearn.pipeline import Pipeline # pipeline
import pandas as pd # pandas for data manipulation

In [2]:
#load cs to dataframe and view 1st 5 rows using head...
hotel_rev = pd.read_csv('HotelRevHelpfulness.csv')
hotel_rev.head()

,aveHelpfulnessRatioUser,stdevHelpfulnessRatioUser,pcReviewsExceedMinHelpfulnessSupport,numReviewsUser,numReviewsHotel,ratingUser,numberSubRatingsUser,subRatingMeanUser,subRatingStdevUser,aveRatingUser,...,completeness_2,completeness_3,numberTermsEntry,percentageAlphaCharsEntry,fractionUpperCaseCharsEntry,fractionYouVsIEntry,numberTermsSummaryQuote,percentageAlphaCharsSummaryQuote,fractionUpperCaseCharsSummaryQuote,reviewHelpfulness
0,1.000000,0.000000,0.666667,3,16,5,4,4.000000,0.000000,4.333333,...,0,1,182,0.788474,0.025703,0.500000,6,0.815789,0.096774,1
1,0.772487,0.377321,0.500000,12,233,5,0,0.000000,0.000000,4.333333,...,0,0,158,0.791888,0.012594,0.500000,1,1.000000,0.083333,1
2,0.715473,0.300437,0.833333,12,302,4,7,3.714286,0.755929,4.166667,...,0,3,59,0.799639,0.024831,0.333333,4,0.828571,0.034483,0
3,0.521250,0.481675,0.222222,36,6,1,4,1.000000,0.000000,3.527778,...,0,0,95,0.782212,0.029155,0.500000,2,0.800000,0.062500,0
4,0.603175,0.246926,1.000000,2,271,3,0,0.000000,0.000000,3.500000,...,0,0,43,0.805128,0.028662,0.000000,1,1.000000,0.142857,0


In [ ]:
# knwo from previosu labs the variables are x and y
# reviw helpfulness column which has target labels.... .values will eb retrieved in y variable amd popped from dataframe... has been removed into y
y = hotel_rev.pop('reviewHelpfulness').values
# assign X to remainder of features... all columns used as features.. except y column we popped out earlier
X = hotel_rev.values

In [ ]:
# splti data into traing and test...
# test size is 50%  / 50%
# random state to ensure same output
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=1/2,
                                                    random_state=42)
X_train.shape, X_test.shape

((243, 23), (243, 23))

In [ ]:
# bild pipeline then define the steps....
# thsi is our pipeline....#
# tsi inlcudes the pre-processing and training...
kNNpipe  = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('kNN', KNeighborsClassifier())])

# Parameters for kNN are prefixed with kNN__
# is tuning the scalaer itself...not just traingi the kNN hyperparameters...
# use different scalign methods....
# StandardScaler() → standardise (mean=0, std=1)
# MinMaxScaler() → scale to [0,1]
# 'passthrough' → no scaling at all — use raw data
# So the grid search will try all three scaling options combined with all kNN parameter combinations.
param_grid = {'scaler':[StandardScaler(), MinMaxScaler(),'passthrough'], 
              'kNN__n_neighbors':[1,3,5,7],
              # how much influence a neighbour has i.e. 
              # "uniform" = all k neighbours get equal vite regardless of how far away
              # "distance" = closer neighbours get more influence - weighted by 1/distance so very close neighbour counts more than a distane on...
              # metric  = how yo measure distance between two points i.e. Euclidean, Manhattan, Minkowski
              # metric is not being defined here so it defaults to minkowski with p=2 (which is euclidean) ' if p=1 => ( its manhattan)
              'kNN__weights':['uniform','distance']
             }
# The double underscore __ is how sklearn knows which step in the pipeline these parameters belong to. The name kNN matches the name given in the pipeline steps

In [ ]:

# create grid search cross validation  -- jus create instance for thsi package and call pipleine and grod and parameters...
grid_search = GridSearchCV(kNNpipe, param_grid=param_grid, verbose = 1)
# train using trainign fearyres and traing target labe
# we dont' need to do anythign except use grid search xval and traing it using x and y train...
grid_search = grid_search.fit(X_train,y_train)

# Runs all 120 fits (24 combinations × 5 folds)
# Scores each combination
# Internally stores the best performing combination

#Fitting 5 folds for each of 24 candidates, totalling 120 fits
# model is trained 120 times....


Fitting 5 folds for each of 24 candidates, totalling 120 fits


In [ ]:
# After running all 120 fits, this returns a dictionary showing which combination of parameters performed best across the cross-validation folds.
grid_search.best_params_

# Other useful attributes alongside best_params_:
# grid_search.best_score_      # the CV score of the best combination
# grid_search.best_estimator_  # the actual fitted best model, ready to predict
#  grid_search.cv_results_      # full results table for all 24 combinations

# So best_params_ answers the question "what settings should I use?" and best_estimator_ gives you the actual model ready to use for predictions.

{'kNN__n_neighbors': 5, 'kNN__weights': 'uniform', 'scaler': 'passthrough'}

### All grid search results
The parameter `cv_results_` gives us access to results on all options tested.  
We store the results in a data frame and print the important information. 

In [ ]:
# this will retrieve all parameters - thsi gives yo all results 
# puts it into a table from a dictionaary
scores_df = pd.DataFrame(grid_search.cv_results_)
# then get best results... based on accuracy which is !rank_test_score!
# drop theb index column aswell
scores_df = scores_df.sort_values(by=['rank_test_score']).reset_index(drop='index')
# names for columns in thr data frame
scores_df [['rank_test_score', 'mean_test_score', 'param_kNN__n_neighbors', 
            'param_kNN__weights','param_scaler']]

# we have 24 classifiers of the k-NN'

,rank_test_score,mean_test_score,param_kNN__n_neighbors,param_kNN__weights,param_scaler
0,1,0.695748,5,uniform,passthrough
1,2,0.683503,5,distance,passthrough
2,3,0.683163,7,uniform,passthrough
3,3,0.683163,3,uniform,passthrough
4,5,0.679167,7,distance,passthrough
5,6,0.674915,3,distance,passthrough
6,7,0.654507,7,uniform,MinMaxScaler()
7,7,0.654507,7,distance,MinMaxScaler()
8,9,0.650340,7,distance,StandardScaler()
9,9,0.650340,7,uniform,StandardScaler()


## Grid Search for Naive Bayes

now apply for Naive Bauyes classifier....
**Q1**  
Repeat the exercise above to fit a Naive Bayes model.  
Consider the same scaling options and `GaussianNB` and `BernoulliNB` as classifier options. 

In [11]:
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [ ]:
NBpipe  = Pipeline(steps=[
    ('scaler', 'passthrough'),
    ('naive_bayes', GaussianNB())])


param_grid = {'scaler':[StandardScaler(), MinMaxScaler(),'passthrough'], 
              'naive_bayes':[GaussianNB(), BernoulliNB()]

             }


In [13]:
grid_search = GridSearchCV(NBpipe, param_grid=param_grid, cv=10, verbose = 1)
# train using trainign fearyres and traing target labe
# we dont' need to do anythign except use grid search xval and traing it using x and y train...
grid_search = grid_search.fit(X_train,y_train)

Fitting 10 folds for each of 6 candidates, totalling 60 fits


In [14]:
grid_search.best_params_

{'naive_bayes': BernoulliNB(), 'scaler': StandardScaler()}

In [ ]:
# create a df to save the results form each instance the results... so we cna see them.. as earlier
scores_df = pd.DataFrame(grid_search.cv_results_)
# sort using 'by'
scores_df = scores_df.sort_values(by=['rank_test_score']).reset_index(drop='index')
scores_df [['rank_test_score', 'param_naive_bayes', 'mean_test_score', 'param_scaler']]


,rank_test_score,param_naive_bayes,mean_test_score,param_scaler
0,1,BernoulliNB(),0.655167,StandardScaler()
1,2,BernoulliNB(),0.646500,MinMaxScaler()
2,3,GaussianNB(),0.642500,StandardScaler()
3,3,GaussianNB(),0.642500,MinMaxScaler()
4,5,BernoulliNB(),0.638500,passthrough
5,6,GaussianNB(),0.638500,passthrough


## Grid Search for Decision Trees
**Q2**  
Find the best decision tree model for the `HotelRevHelpfulness` dataset considering  `max_leaf_nodes` and the splitting `criterion`. The splitting `criterion` can be either 'gini' or 'entropy', you can select your own options for `max_leaf_nodes`.

In [18]:
from sklearn.tree import DecisionTreeClassifier

There are no pre-processing steps so there is no need for a pipeline.
We go with [3, 5, 10, 20, 50] as the options for max_leaf_nodes.. try different numbers... depends on the dataset yuo have..

DTs have a number of ndoes... and decision is entropy or gini...


In [ ]:
tree_grid = {'criterion':['gini', 'entropy'], 'max_leaf_nodes':[3, 5, 10, 20, 50],}



In [ ]:
# create DT classifier as an instance
tree = DecisionTreeClassifier()

# pass that instance to the grid search pipeline
tree_search = GridSearchCV(tree, param_grid=tree_grid, cv=10, verbose = 1)
# train using trainign fearyres and traing target labe
# we dont' need to do anythign except use grid search xval and traing it using x and y train...
# this means the instacne will be trained using target variables just for trainnig
tree_search = tree_search.fit(X_train,y_train)

Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [ ]:
# whatare the best aprameters
tree_search.best_params_

{'criterion': 'gini', 'max_leaf_nodes': 5}

The main message we cna take from looking at all the results in that less leaf nodes is inclined to be better.
This suggests that bushier trees are inclined to oevrfit

In [25]:
# create a df to save the results form each instance the results, all results.... so we cna see them.. as earlier
scores_df = pd.DataFrame(tree_search.cv_results_)
# sort using 'by'
scores_df = scores_df.sort_values(by=['rank_test_score']).reset_index(drop='index')
scores_df [['rank_test_score', 'param_criterion', 'mean_test_score', 'param_max_leaf_nodes']]

,rank_test_score,param_criterion,mean_test_score,param_max_leaf_nodes
0,1,gini,0.695167,5
1,2,entropy,0.679500,5
2,3,gini,0.678833,3
3,4,gini,0.674667,10
4,5,entropy,0.658333,10
5,6,gini,0.654333,20
6,7,entropy,0.650833,3
7,8,entropy,0.650833,20
8,9,entropy,0.642500,50
9,10,gini,0.629333,50


## Model Selectoion

**Q3**  
Which model would you recommend for this dataset?

**Ans**
Its a toss up between a Decision Tree using the gini splitting criterion and max_leaf_nodes = 5 and a k-NN classifier with k=5, uniform weighting and no scaling

DT and k-NN have same test score..
check **mean_test_score** values... and compare across them all...
- 1st k-NN = 0.69.. we have weighting ad scaling
- 2nd NB = 0.65...
- 3rd DT = 0.69... DT we dont need to consider weighting or scaling.....